In [3]:
# %% [markdown]
# # 02. 模型训练演示 (Training Pipeline Demo)
# 
# **目标**: 
# 1. 调用 `src` 模块加载数据。
# 2. 训练 LSTM 模型。
# 3. 评估模型性能并可视化预测结果。
# 
# **前提**: 确保 `src/` 文件夹下的代码已更新且没有语法错误。

# %%
import sys
import os
import torch
import matplotlib.pyplot as plt
import numpy as np

# 【关键步骤】将项目根目录加入 python 路径，否则无法 import src
# 获取当前 notebook 所在文件夹的上一级目录
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project Root: {project_root}")

# 导入自定义模块
try:
    from src.config import config
    from src.data_loader import get_data_loaders
    from src.model import LSTMModel
    from src.train import train_model
    from src.evaluate import evaluate_model
    from src.utils import plot_loss_curve, plot_predictions
    print("✅ Successfully imported modules from src.")
except ImportError as e:
    print(f"❌ ImportError: {e}")
    print("请检查 src 文件夹下是否有 __init__.py 以及相关 .py 文件")

# 检查设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# %% [markdown]
# ## 1. 准备数据 (Data Preparation)

# %%
# 这里你可以动态修改 sequence_length 来测试不同窗口大小的效果
seq_len = 30  
batch_size = 32

print(">>> Generating Data Loaders...")
# 调用 src.data_loader 中的函数
train_loader, test_loader, scaler = get_data_loaders(
    sequence_length=seq_len, 
    batch_size=batch_size
)

print(f"Train Batches: {len(train_loader)}")
print(f"Test Batches: {len(test_loader)}")

# %% [markdown]
# ## 2. 初始化模型 (Initialize Model)

# %%
# 临时修改 config 中的参数 (可选)
config.device = device
config.epochs = 50   # 演示用 50 epochs 即可
config.learning_rate = 0.001

model = LSTMModel().to(device)
print(model)

# %% [markdown]
# ## 3. 训练模型 (Training)
# 调用 train_model 函数。如果你的 src/train.py 还没有修改返回值，这里可能会报错。
# 期望返回值: model, train_losses, val_losses

# %%
print(">>> Starting Training...")

# 调用训练函数 (设置 use_wandb=False 避免 notebook 弹出一堆登录提示)
model, train_losses, val_losses = train_model(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    config=config,
    device=device,
    use_wandb=False 
)

# 绘制 Loss 曲线
plt.figure(figsize=(10,5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Training Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.show()

# %% [markdown]
# ## 4. 模型预测与评估 (Evaluation)

# %%
print(">>> Evaluating on Test Set...")

# 使用 evaluate_model 模块 (或者直接在这里写预测逻辑)
metrics, predictions, actuals = evaluate_model(model, test_loader, device)

print(f"Test RMSE: {metrics['rmse']:.4f}")
print(f"Test MAE: {metrics['mae']:.4f}")

# %% [markdown]
# ## 5. 可视化预测结果 (Visualization)
# 展示一部分测试集数据的预测效果。

# %%
# 只取前 200 个时间步进行展示，太密了看不清
limit = 200

plt.figure(figsize=(12, 6))
plt.plot(actuals[:limit], label='Actual NDVI', color='green', alpha=0.7)
plt.plot(predictions[:limit], label='Predicted NDVI', color='red', linestyle='--', alpha=0.9)
plt.title(f'NDVI Prediction (First {limit} days of Test Set)')
plt.xlabel('Time Step (Days)')
plt.ylabel('Normalized NDVI')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

Project Root: d:\Grade3Semester1\PnAofBigdata\期末项目\Drought_Prediction_LSTM
❌ ImportError: cannot import name 'get_data_loaders' from 'src.data_loader' (d:\Grade3Semester1\PnAofBigdata\期末项目\Drought_Prediction_LSTM\notebooks\..\src\data_loader.py)
请检查 src 文件夹下是否有 __init__.py 以及相关 .py 文件
Using device: cpu
>>> Generating Data Loaders...


NameError: name 'get_data_loaders' is not defined